# Summary

Test the Google RAG components

### Use the ADK environment


In [14]:
import os, sys

from vertexai import rag
from vertexai.generative_models import GenerativeModel, Tool
import vertexai

utils_path = "../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
import gcp_tools as gct

# Set environment variables
dotenv_path = "../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()

# Initialize Vertex AI API once per session
vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
              location=os.environ["GOOGLE_CLOUD_LOCATION"],
              staging_bucket=os.environ["STAGING_BUCKET"])




## Explore existing Corpora (note these are saved in GCP)

In [12]:
# See what methods are available in the rag object
# dir(rag)

# Find existing corpora and delete
existing_corpora = rag.list_corpora()
# print(existing_corpora)

# Get the corpus names and put in a list
existing_corpora_list = []
for existing_corpus in existing_corpora:
    existing_corpora_list.append(existing_corpus.name)
print(existing_corpora_list)

# Delete these
# for corpus_name in existing_corpora_list:
#     rag.delete_corpus(corpus_name)

# Check that there is nothing there
# for existing_corpus in existing_corpora:
#     existing_corpora_list.append(existing_corpus.name)


['projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/1290281293241647104']
Successfully deleted the RagCorpus.


## Get the contents of the bucket that has text files

In [15]:
# Get a list of contents in the GCP bucket that will be used for corpus
gcs_bucket_name = "ccc-crawl_data_xb"
gcs_path = "crawl_data/text_files"
bcontents = gct.gcp_list_bucket(gcp_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                gcs_bucket_name=gcs_bucket_name,
                                path=gcs_path)
print(bcontents)


['crawl_data/text_files/', 'crawl_data/text_files/calmattersdigitaldemocracyorg_2025May01_text.txt', 'crawl_data/text_files/cccaoeorg_2025May01_text.txt', 'crawl_data/text_files/ccrctccolumbiaedu_2025May01_text.txt', 'crawl_data/text_files/enwikipediaorg_2025May01_text.txt', 'crawl_data/text_files/icangotocollegecom_2025May01_text.txt', 'crawl_data/text_files/laocagov_2025May01_text.txt', 'crawl_data/text_files/nscresearchcenterorg_2025May01_text.txt', 'crawl_data/text_files/wwwaaccncheedu_2025May01_text.txt', 'crawl_data/text_files/wwwccccoedu_2025May01_text.txt', 'crawl_data/text_files/wwwccleagueorg_2025May01_text.txt', 'crawl_data/text_files/wwwecsorg_2025May01_text.txt']


## Create and save a corpus

In [18]:
# Create an embedding model
embedding_model_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model="publishers/google/models/text-embedding-005"
    )
)


In [19]:
# Create a corpus
display_name = "ccc_test_corpus"
rag_corpus = rag.create_corpus(
    display_name=display_name,
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_model_config
    ),
)


In [20]:
# Import Files to the RagCorpus
paths = ["gs://{}/{}".format(gcs_bucket_name, gcs_path)]
rag.import_files(
    rag_corpus.name,
    paths,
    # Optional
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=512,
            chunk_overlap=100,
        ),
    ),
    max_embedding_requests_per_min=1000,  # Optional
)


imported_rag_files_count: 10
failed_rag_files_count: 1

## If needed look for the file(s) that failed

In [34]:
# find the file that failed
corpus_files = rag.list_files(rag_corpus.name)

# Get the displaynmes
corpus_file_display_names = [f.display_name for f in corpus_files]

# Check what's missing
bcontents_files = [os.path.split(bc)[1] for bc in bcontents]
missing_files = [f for f in bcontents_files if f not in corpus_file_display_names]
missing_files


['', 'wwwaaccncheedu_2025May01_text.txt']

## Get the corpus name

In [35]:
# Check that the new corpus is there
existing_corpora = rag.list_corpora()
existing_corpora_list = []
for existing_corpus in existing_corpora:
    existing_corpora_list.append(existing_corpus.name)

existing_corpora_list


['projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/4611686018427387904']